In [ ]:
import sys
sys.path.append('/data/users/quentin/final_package/')
import scClone2DR as sccdr
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
import pickle
import torch
from tqdm import tqdm
import os
import plotly.io as pio
np.float_ = np.float64
rootpath = '/data/users/quentin/final_package/experiments/paper_results'

COHORT = 'aml'
gene_set_collections = ['geneOncoKB','gene','c6','hallmarks', 'c2_pid']#, 'c2_kegg_medicus']
cohort2clonemodes = {'melanoma': ["scatrex",'scatrex_rawcounts_scvi', 'phenograph'], 'aml':['phenograph']}
gene_set_collection = gene_set_collections[3]
clonemode = cohort2clonemodes[COHORT][0]
mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)

directory = os.path.join(rootpath,COHORT)
if not os.path.exists(directory):
    os.makedirs(directory)
pathsave = os.path.join(rootpath,COHORT,gene_set_collection+'_'+clonemode)
# if not os.path.exists(pathsave):
#     os.makedirs(pathsave)
    
if COHORT=='melanoma':
    path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
    path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
    concentration_DMSO = '100'
    concentration_drug = '5'
    path_info_cohort = None
elif COHORT=='aml':
    concentration_DMSO = '200'
    concentration_drug = '10'
    path_rna = '/data/users/04_share_reanalysis_results/aml_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
    path_fastdrug = '/data/users/04_share_reanalysis_results/01_aml/AML_PCY_cell_numbers_no_plate_effect_correction.csv'
    path_info_cohort = '/data/users/04_share_reanalysis_results/01_aml/2024-08-15_aml_overview_scRNA.tsv'
model = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort, type_guide="lowrank_MVN", rank=10)
data_ref = model.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)


if False:
    idxs_train = [i for i in range(int(data_ref['N']))]
    idxs_test = []

    data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, idxs_test)
    params_svi = model.train(data_train, penalty_l1=0.3, penalty_l2=0.3 , n_steps=30)
    with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'wb') as handle:
        pickle.dump(params_svi, handle, protocol=pickle.HIGHEST_PROTOCOL)
else:
    with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
        params_svi = pickle.load(handle)

In [ ]:
np.sort(model.FD.selected_drugs)

In [ ]:
if False:
    idxs_train = [i for i in range(int(data_ref['N']))]
    idxs_test = []

    data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, idxs_test)
    params_svi = model.train(data_train, penalty_l1=4, penalty_l2=4, n_steps=2000)

    #params_svi = model.train(data_train, penalty_l1=3.2, penalty_l2=3.2 , n_steps=2000)
    #params_svi = model.train(data_train, penalty_l1=0.1, penalty_l2=0.1 , n_steps=2000)
    with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'wb') as handle:
        pickle.dump(params_svi, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
pathsave =  os.path.join(rootpath,COHORT,gene_set_collection+'_'+clonemode)
pathsave

In [ ]:
posterior_mean_params = model.sampling_from_posterior(data_ref, pathsave, params=params_svi, nb_ites=1000, sample_names=model.sample_names)

In [ ]:
if False:
    with open(os.path.join(rootpath,'{0}/{1}_{2}/posterior_mean_params.pkl'.format(COHORT, gene_set_collection, clonemode)), 'wb') as handle:
        pickle.dump(posterior_mean_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
res = posterior_mean_params

In [ ]:
params_svi['beta'].shape, data_ref['X'].shape

In [ ]:
import seaborn as sns
sns.clustermap(params_svi['beta'][:,:])

In [ ]:
plt.plot(params_svi['beta'][10,:])
plt.xlabel('Genes', fontsize=14)
plt.title('$\\beta$ coefficients for the drug: '+model.FD.selected_drugs[10])
plt.show()

In [ ]:
x = params_svi['beta'][10,:]
x = np.sort(x)
plt.plot(x, np.cumsum(np.ones(len(x))))
plt.ylabel('Genes', fontsize=14)
plt.xlabel('$\\beta$ coefficients for the drug: '+model.FD.selected_drugs[10], fontsize=14)
plt.show()

In [ ]:
plt.imshow(data_ref['n_rna'])

In [ ]:
params_svi['proportions'].shape

In [ ]:
with open(os.path.join(rootpath,'{0}/{1}_{2}/posterior_mean_params.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
    res = pickle.load(handle)

In [ ]:
sccdr.resultanalysis.show_fractions(data_ref, res, idxdrug=0)

In [ ]:
sccdr.resultanalysis.show_cells(data_ref, res)

In [ ]:
sccdr.resultanalysis.scatter_counts(data_ref, res, color_mode='patient')

In [ ]:
plt.imshow(data_ref['masks']['RNA'])

In [ ]:

import seaborn as sns
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
def survival_probabilities(data, pi, cluster2clonelabel, df_info_cohort=None, idxdrug=0, label_colors=None, drug_name=None, clustername='subclone', savefig=None):
    # Comparing true survival probabilities and the one estimated
    
    label2name = {'healthy':'Non-malignant', 'tumor':'Tumor', 'putative':'Putative'}
    
    for d in range(data['D']):
        pi[d,:,:][~(data['masks']['RNA'])] = float('nan')
        pi[d,:,:][~(data['masks']['RNA'])] = float('nan')
    
    # Sort clusters based on their labels
    sorted_indices = np.arange(len(cluster2clonelabel))#np.argsort(cluster2clonelabel)
    sorted_pi = pi[idxdrug, sorted_indices, :]
    
    if not(df_info_cohort is None):
        # Sort samples by patient_id and tissue_type
        sample_names = df_info_cohort.index.values
        sorted_meta_data = df_info_cohort.sort_values(by=['patient_id', 'tissue_type'])
        sorted_sample_ids = sorted_meta_data.index
        sample2idx = {sample: idx for idx, sample in enumerate(sorted_sample_ids.values)}
        sorted_indices_patient = np.array([sample2idx[sample] for sample in sample_names])
        sorted_pi = sorted_pi[:, sorted_indices_patient]
    
    # Get unique cluster labels and assign a color to each
    unique_labels = np.unique(cluster2clonelabel)
    if label_colors is None:
        label_colors = plt.colormaps['tab20'](np.linspace(0, 1, len(unique_labels)))

    
    # Create a color array for the y-axis labels, formatted for display
    row_colors = np.array([label_colors[np.where(unique_labels == label)[0][0]] for label in np.array(cluster2clonelabel)[sorted_indices]])
    
    # Reshape to a 2D array of RGB colors (n_clusters, 3) -> (n_clusters, 1, 3) for displaying as an image
    row_colors = row_colors.reshape(-1, 1, 4)  # Change 3 to 4 if the color has an alpha channel (RGBA)
    
    if not(df_info_cohort is None):
        # Create the main plot with two subplots: one for the color bar and one for the heatmap
        fig = plt.figure(layout="constrained")
        gs = GridSpec(2, 2, figure=fig, width_ratios=[0.05, 1], height_ratios=[1, 0.05])
        ax1 = fig.add_subplot(gs[0, 0])
        ax2 = fig.add_subplot(gs[0, 1])

    else:
        # Create the main plot with two subplots: one for the color bar and one for the heatmap
        fig, (ax1, ax2) = plt.subplots(ncols=2, gridspec_kw={"width_ratios": [0.005, 1]}, figsize=(12, 8))

    # Plot the color bar on the left using the label colors
    ax1.imshow(row_colors, aspect='auto', interpolation='nearest')
    ax1.axis('off')  # Remove the axis for the color bar
    
    # Plot the heatmap using seaborn
    sns.heatmap(sorted_pi, cmap='viridis', cbar=True, ax=ax2, yticklabels=False, linewidths=0.5,linecolor='lightgray')
    
    # Set the y-axis labels to the sorted cluster indices
    ax2.set_yticks(np.arange(len(sorted_indices)) + 0.5)

    # Custom legend for cluster labels
    handles = [mpatches.Patch(color=label_colors[i], label=f'{label2name[label]} {clustername}') for i, label in enumerate(unique_labels)]
    ax2.legend(handles=handles, bbox_to_anchor=(0.5, 1.2), loc='upper center', borderaxespad=0., ncols=1)
    ax2.set_xticks([])

    # Optional: If you have patient labels
    # ax_heatmap.set_xticks(ticks=np.arange(sorted_pi.shape[1]))
    # ax_heatmap.set_xticklabels(patient_labels, rotation=90)
    
    # Show the plot
    if not(drug_name is None):
        if drug_name=='empty':
            pass
        else:
            plt.suptitle(f"Heatmap for the drug: {drug_name}")
    else:
        plt.suptitle(f"Heatmap for Drug Index {idxdrug}")

    if not(df_info_cohort is None):
        ax3 = fig.add_subplot(gs[1, 1])
        # Get unique cluster labels and assign a color to each
        unique_labels = np.unique(sorted_meta_data['patient_id'].values)
        label_colors = plt.colormaps['tab20'](np.linspace(0, 1, len(unique_labels)))
        # Create a color array for the y-axis labels, formatted for display
        col_colors = np.array([label_colors[np.where(unique_labels == label)[0][0]] for label in np.array(sorted_meta_data['patient_id'].values)])

        # Reshape to a 2D array of RGB colors (n_clusters, 3) -> (n_clusters, 1, 3) for displaying as an image
        col_colors = col_colors.reshape(1, -1, 4)  # Change 3 to 4 if the color has an alpha channel (RGBA)
        ax3.imshow(col_colors, aspect='auto', interpolation='nearest')
        #ax3.axis('off')  # Remove the axis for the color bar
        ax3.set_xlabel('Samples')
        ax3.set_xticks([])
        ax3.set_yticks([])
    if not(savefig is None):
        plt.savefig(savefig, dpi=250, bbox_inches='tight')
    plt.show()

In [ ]:
plt.imshow(res['pi'][0,:,:])

In [ ]:
for i in range(70):
    plt.imshow(data_ref['masks']['SingleCell'][:,i,:])
    plt.show()

In [ ]:
model.dfclones

In [ ]:
for idxdrug in range(70):
    if COHORT=='melanoma' and model.FD.selected_drugs[idxdrug]=='Carfilzomib':
        label_colors = [[0.255, 0.412, 0.882, 1.0], [0.863, 0.078, 0.235, 1.0]]
        sccdr.resultanalysis.survival_probabilities(data_ref, res['pi'].detach().numpy(), model.cluster2clonelabel, idxdrug=idxdrug)
    elif COHORT=='aml':
        survival_probabilities(data_ref, res['pi'].detach().clone().numpy(), model.cluster2clonelabel, df_info_cohort=model.dfinfo, idxdrug=idxdrug, drug_name='empty', savefig='aml_c6_{0}.png'.format(model.FD.selected_drugs[idxdrug]))

In [ ]:
def survival_probabilities_relative_by_patient_optimized(data, pi, props, cluster2clonelabel, cluster2cat, cat2clusters, selected_drugs, idxs2drugs=None, df_info_cohort=None, sampleID=0, savefig=None):
    from copy import deepcopy
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
    from matplotlib.gridspec import GridSpec
    if idxs2drugs is None:
        idxs2drugs = range(data['D'])
    ratio_pi = deepcopy(pi)
    for idxdrug in range(pi.shape[0]):
        for sampleid in range(pi.shape[2]):
            learned_props = props[sampleid, :]
            healthy_survival = np.sum(
                np.nan_to_num(pi[idxdrug, :, sampleid] * learned_props)[cat2clusters['healthy']]
            )
            healthy_survival /= np.sum(learned_props[cat2clusters['healthy']])
            ratio_pi[idxdrug, :, sampleid] /= healthy_survival

    scores = np.sum(
        np.nan_to_num(ratio_pi)[:, cat2clusters['tumor'], :] * 
        (props[:, cat2clusters['tumor']].T)[None, :, :], axis=1
    )
    scores /= np.tile((np.sum(props[:, cat2clusters['tumor']], axis=1)).reshape(1, -1), (ratio_pi.shape[0], 1))
    scores = scores[:, sampleID]
    scores = np.clip(scores, a_min=0, a_max=10)

    for d in range(data['D']):
        pi[d, :, :][~(data['masks']['RNA'])] = float('nan')
        ratio_pi[d, :, :][~(data['masks']['RNA'])] = float('nan')

    available_clusters = np.array([
        i for i in range(pi.shape[1]) 
        if (not np.isnan(pi[0, i, sampleID]) and (cluster2cat[i] != 'healthy'))
    ]).astype(int)
    pi = pi[:, available_clusters, :]
    ratio_pi = ratio_pi[:, available_clusters, :]

    learned_props = props[sampleID, available_clusters]
    cluster2clonelabel = np.array(cluster2clonelabel)[available_clusters]
    cluster2cat = np.array(cluster2cat)[available_clusters]
    
    sorted_indices = np.argsort(learned_props)
    cluster2cat = cluster2cat[sorted_indices]
    learned_props = learned_props[sorted_indices]
    cluster2clonelabel = np.array(cluster2clonelabel)[sorted_indices]
    sorted_pi = pi[:, sorted_indices, sampleID]
    sorted_ratio_pi = ratio_pi[:, sorted_indices, sampleID]

    idxs_sort_drugs = np.argsort(scores)
    sorted_drugs = selected_drugs[idxs_sort_drugs]
    sorted_pi = sorted_pi[idxs_sort_drugs, :]
    sorted_ratio_pi = sorted_ratio_pi[idxs_sort_drugs, :]
    sorted_scores = scores[idxs_sort_drugs]
    
    past2new = {past:new for new, past in enumerate(idxs_sort_drugs)}
    idxs2drugs = np.sort([past2new[past] for past in idxs2drugs])

    label_colors = plt.colormaps['Oranges'](np.linspace(0, 1, 100))
    row_colors = np.array([label_colors[int(prop * 100)] for prop in learned_props])
    row_colors = row_colors.reshape(-1, 1, 4)

    fig = plt.figure(layout="constrained", figsize=(15, 8))
    gs = GridSpec(2, 2, figure=fig, height_ratios=[1, 0.1], width_ratios=[0.05, 1])

    ax1 = fig.add_subplot(gs[0, 0])
    # Replace ax1 with a horizontal bar chart
    ax1.clear()
    ax1.barh(
        y=np.arange(len(learned_props)), 
        width=learned_props, 
        color='orange', 
        edgecolor='black', 
        align='center', 
        height=0.9
    )
    ax1.set_yticks(np.arange(len(learned_props)))
    ax1.set_yticklabels([])
    ax1.invert_yaxis()  # Invert y-axis to match heatmap row order
    ax1.set_xlim(0, max(learned_props) * 1.1)  # Add padding to bar chart
    ax1.xaxis.set_label_position('top')
    ax1.xaxis.tick_top()
    ax1.set_xlabel("Cluster size", fontsize=14, loc='left')
    #ax1.tick_params(axis='x', labelsize=13)
    ax1.set_xticks([])
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    for i, prop in enumerate(learned_props):
        ax1.text(
            prop * 1.02,  # Position the text slightly to the right of the bar
            i,  # Align with the bar
            f"{prop:.2f}",  # Display the value with 2 decimal places
            va='center',  # Vertically center the text
            fontsize=12,  # Font size for readability
            color='black'  # Text color
        )


    ax2 = fig.add_subplot(gs[0, 1])
    colors = [(0, 0, 1), (0.5, 0.5, 0.5), (1, 0, 0)]
    custom_cmap = LinearSegmentedColormap.from_list("terrain", colors)
    vmin = min([0.9, np.min(np.nan_to_num(sorted_ratio_pi[idxs2drugs,:]))])
    vmax = max([1.1, np.max(np.nan_to_num(sorted_ratio_pi[idxs2drugs,:]))])
    norm = TwoSlopeNorm(vmin=vmin, vmax=vmax, vcenter=1)
    sns.heatmap(sorted_ratio_pi.T, cmap=custom_cmap, norm=norm, cbar=True, alpha=0.7, ax=ax2, yticklabels=False)
    ax2.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    ax2.set_ylabel('Tumor clusters', fontsize=14)
    colorbar = ax2.collections[0].colorbar
    colorbar.set_label("Relative cluster\nsurvival probability", fontsize=12, labelpad=10)

    
    sorted_scores_filtered = sorted_scores[idxs2drugs]
    ax3 = fig.add_subplot(gs[1, 1], sharex=ax2)
    idxs = np.where(sorted_scores_filtered<=1)[0]
    ax3.plot(np.arange(len(sorted_scores_filtered))[idxs], sorted_scores_filtered[idxs], marker='o', color='blue')
    idxs = np.where(sorted_scores_filtered>1)[0]
    ax3.plot(np.arange(len(sorted_scores_filtered))[idxs], sorted_scores_filtered[idxs], marker='o', color='red')
    ax3.set_xticks([])
    ax3.set_xticks(np.arange(len(sorted_drugs[idxs2drugs])))
    ax3.set_xticklabels(sorted_drugs[idxs2drugs], rotation=90, fontsize=13)
    ax3.set_ylabel("Relative tumor\nsurvival probability", fontsize=14)
    ax3.grid(axis='y', linestyle='--', alpha=0.7)


    plt.suptitle(f"Heatmap for sample: {sampleID}", ha='center', fontsize=18)

    if savefig:
        plt.savefig(savefig, dpi=250, bbox_inches='tight')
    plt.show()
    

In [ ]:
from matplotlib.colors import FuncNorm



def survival_probabilities_relative_by_patient_optimized(data, pi, props, cluster2clonelabel, cluster2cat, cat2clusters, selected_drugs, idxs2drugs=None, df_info_cohort=None, sampleID=0, savefig=None):
    from copy import deepcopy
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, Normalize
    from matplotlib.gridspec import GridSpec
    if idxs2drugs is None:
        idxs2drugs = range(data['D'])
    ratio_pi = deepcopy(pi)
    for idxdrug in range(pi.shape[0]):
        for sampleid in range(pi.shape[2]):
            learned_props = props[sampleid, :]
            healthy_survival = np.sum(
                np.nan_to_num(pi[idxdrug, :, sampleid] * learned_props)[cat2clusters['healthy']]
            )
            healthy_survival /= np.sum(learned_props[cat2clusters['healthy']])
            ratio_pi[idxdrug, :, sampleid] /= healthy_survival

    scores = np.sum(
        np.nan_to_num(ratio_pi)[:, cat2clusters['tumor'], :] * 
        (props[:, cat2clusters['tumor']].T)[None, :, :], axis=1
    )
    scores /= np.tile((np.sum(props[:, cat2clusters['tumor']], axis=1)).reshape(1, -1), (ratio_pi.shape[0], 1))
    scores = scores[:, sampleID]
    scores = np.clip(scores, a_min=0, a_max=10)

    for d in range(data['D']):
        pi[d, :, :][~(data['masks']['RNA'])] = float('nan')
        ratio_pi[d, :, :][~(data['masks']['RNA'])] = float('nan')

    available_clusters = np.array([
        i for i in range(pi.shape[1]) 
        if (not np.isnan(pi[0, i, sampleID]) and (cluster2cat[i] != 'healthy'))
    ]).astype(int)
    pi = pi[:, available_clusters, :]
    ratio_pi = ratio_pi[:, available_clusters, :]

    learned_props = props[sampleID, available_clusters]
    cluster2clonelabel = np.array(cluster2clonelabel)[available_clusters]
    cluster2cat = np.array(cluster2cat)[available_clusters]
    
    sorted_indices = np.argsort(learned_props)
    cluster2cat = cluster2cat[sorted_indices]
    learned_props = learned_props[sorted_indices]
    cluster2clonelabel = np.array(cluster2clonelabel)[sorted_indices]
    sorted_pi = pi[:, sorted_indices, sampleID]
    sorted_ratio_pi = ratio_pi[:, sorted_indices, sampleID]

    idxs_sort_drugs = np.argsort(scores)
    sorted_drugs = selected_drugs[idxs_sort_drugs]
    sorted_pi = sorted_pi[idxs_sort_drugs, :]
    sorted_ratio_pi = sorted_ratio_pi[idxs_sort_drugs, :]
    sorted_scores = scores[idxs_sort_drugs]
    
    past2new = {past:new for new, past in enumerate(idxs_sort_drugs)}
    idxs2drugs = np.sort([past2new[past] for past in idxs2drugs])

    label_colors = plt.colormaps['Oranges'](np.linspace(0, 1, 100))
    row_colors = np.array([label_colors[int(prop * 100)] for prop in learned_props])
    row_colors = row_colors.reshape(-1, 1, 4)

    fig = plt.figure(layout="constrained", figsize=(15, 8))
    gs = GridSpec(2, 2, figure=fig, height_ratios=[1, 0.1], width_ratios=[0.05, 1])

    ax1 = fig.add_subplot(gs[0, 0])
    # Replace ax1 with a horizontal bar chart
    ax1.clear()
    ax1.barh(
        y=np.arange(len(learned_props)), 
        width=learned_props, 
        color='orange', 
        edgecolor='black', 
        align='center', 
        height=0.9
    )
    ax1.set_yticks(np.arange(len(learned_props)))
    ax1.set_yticklabels([])
    ax1.invert_yaxis()  # Invert y-axis to match heatmap row order
    ax1.set_xlim(0, max(learned_props) * 1.1)  # Add padding to bar chart
    ax1.xaxis.set_label_position('top')
    ax1.xaxis.tick_top()
    ax1.set_xlabel("Cluster size", fontsize=14, loc='left')
    #ax1.tick_params(axis='x', labelsize=13)
    ax1.set_xticks([])
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    for i, prop in enumerate(learned_props):
        ax1.text(
            prop * 1.02,  # Position the text slightly to the right of the bar
            i,  # Align with the bar
            f"{prop:.2f}",  # Display the value with 2 decimal places
            va='center',  # Vertically center the text
            fontsize=12,  # Font size for readability
            color='black'  # Text color
        )


    ax2 = fig.add_subplot(gs[0, 1])
    split = 0.7  # fraction of colorbar used for values < 1
    colors = [
        (0.0, (0, 0, 1)),     # blue
        (split, (1, 1, 1)),   # white at 1
        (1.0, (1, 0, 0))      # red
    ]    
    custom_cmap = LinearSegmentedColormap.from_list("terrain", colors)
    vmin = min([0.9, np.min(np.nan_to_num(sorted_ratio_pi[idxs2drugs,:]))])
    vmax = max([1.1, np.max(np.nan_to_num(sorted_ratio_pi[idxs2drugs,:]))])


    def forward(x):
        x = np.asarray(x)
        y = np.empty_like(x, dtype=float)

        below = x <= 1
        above = x > 1

        y[below] = split * (x[below] - vmin) / (1 - vmin)
        y[above] = split + (1 - split) * (x[above] - 1) / (vmax - 1)

        return y

    def inverse(y):
        y = np.asarray(y)
        x = np.empty_like(y, dtype=float)

        below = y <= split
        above = y > split

        x[below] = vmin + (y[below] / split) * (1 - vmin)
        x[above] = 1 + ((y[above] - split) / (1 - split)) * (vmax - 1)

        return x
    norm = FuncNorm((forward, inverse), vmin=vmin, vmax=vmax)
    sns.heatmap(sorted_ratio_pi.T, cmap=custom_cmap, norm=norm, cbar=True, alpha=0.7, ax=ax2, yticklabels=False)
    ax2.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    ax2.set_ylabel('Tumor clusters', fontsize=14)
    colorbar = ax2.collections[0].colorbar
    
    # Example: more ticks below 1, fewer above
    ticks_below = np.linspace(vmin, 1, 8)      # dense
    ticks_above = np.linspace(1, vmax, 4)[1:]  # sparse, skip duplicate 1

    ticks = np.concatenate([ticks_below, ticks_above])

    colorbar.set_ticks(ticks)
    colorbar.set_ticklabels([f"{t:.2f}" for t in ticks])
    colorbar.set_label("Relative cluster\nsurvival probability", fontsize=12, labelpad=10)

    sorted_scores_filtered = sorted_scores[idxs2drugs]
    ax3 = fig.add_subplot(gs[1, 1], sharex=ax2)
    idxs = np.where(sorted_scores_filtered<=1)[0]
    ax3.plot(np.arange(len(sorted_scores_filtered))[idxs], sorted_scores_filtered[idxs], marker='o', color='blue')
    idxs = np.where(sorted_scores_filtered>1)[0]
    ax3.plot(np.arange(len(sorted_scores_filtered))[idxs], sorted_scores_filtered[idxs], marker='o', color='red')
    ax3.set_xticks([])
    ax3.set_xticks(np.arange(len(sorted_drugs[idxs2drugs])))
    ax3.set_xticklabels(sorted_drugs[idxs2drugs], rotation=90, fontsize=13)
    ax3.set_ylabel("Relative tumor\nsurvival probability", fontsize=14)
    ax3.grid(axis='y', linestyle='--', alpha=0.7)


    plt.suptitle(f"Heatmap for sample: {sampleID}", ha='center', fontsize=18)

    if savefig:
        from pathlib import Path
        pathsavefig = f"/data/users/quentin/package_paper/experiments/figures/{COHORT}/figures_drug_response_geneOncoKB_rawcounts_scvi/"
        Path(pathsavefig).mkdir(parents=True, exist_ok=True)

        plt.savefig(os.path.join(pathsavefig, savefig), dpi=250, bbox_inches='tight')
    plt.show()
    

In [ ]:
np.where(model.sample_names == "MADIBUG")

In [ ]:
res['pi'].detach().numpy().shape

In [ ]:
for sampleID in range(model.N):
    print(sampleID)
    samplename = model.sample_names[sampleID]
    if COHORT=='aml':
        idx_dau = np.where(model.FD.selected_drugs=='Daunorubicin')[0][0]
        idxs2drugs = [i for i in range(len(model.FD.selected_drugs)) if i!=idx_dau]
        drugs = model.FD.selected_drugs
        pi = res['pi'].detach().numpy()
        try:
            survival_probabilities_relative_by_patient_optimized(data_ref, pi, params_svi['proportions'], model.cluster2clonelabel, model.cluster2cat, model.cat2clusters, drugs, idxs2drugs=idxs2drugs, sampleID=sampleID, savefig=f'{samplename}.png')
        except:
            print('sampleID error:', sampleID)
            pass
    else:
        survival_probabilities_relative_by_patient_optimized(data_ref, res['pi'].detach().numpy(), params_svi['proportions'], model.cluster2clonelabel, model.cluster2cat, model.cat2clusters, model.FD.selected_drugs, sampleID=sampleID, savefig=f'{samplename}.png')

# Comparing survival probabilites across different gene sets

In [ ]:
gene_set2pi = {}
gene_set_collectionstemp = ['geneOncoKB']
for gene_set_collection in gene_set_collectionstemp:
    with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
        params_svi = pickle.load(handle)
        props = params_svi['proportions']
        N = props.shape[0]
    with open(os.path.join(rootpath,'{0}/{1}_{2}/posterior_mean_params.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
        res = pickle.load(handle)
        pi = res['pi'].detach().numpy().copy()
        D = pi.shape[0]
        Kmax = pi.shape[1]
    pi_healthy = np.zeros((D,N))
    norm = np.zeros(N)
    for i, label in enumerate(model.cluster2clonelabel):
        pi_healthy += props[:,i][None,:] * np.nan_to_num(pi[:,i,:])
        norm += props[:,i]
    pi_healthy /= np.tile(norm, (D,1))
    pi /= np.tile(pi_healthy[:,None,:], (1,Kmax,1))  
    #pi /= np.max(pi_healthy)  
    gene_set2pi[gene_set_collection] = pi 

    for d in range(data_ref['D']):
        pi[d,:,:][~(data_ref['masks']['RNA'])] = float('nan')
        pi[d,:,:][~(data_ref['masks']['RNA'])] = float('nan')

In [ ]:
model.dfinfo = None

In [ ]:
import copy
#idxdrug = np.where(model.FD.selected_drugs=='Vosaroxin')[0][0]
for gene_set, pi in gene_set2pi.items():
    print(gene_set)
    for idxdrug, drug_name in enumerate(model.FD.selected_drugs):
        global_min = 1
        global_max = 1
        for gene_set, pi in gene_set2pi.items():
            global_min = np.min([global_min, np.nanmin(pi[idxdrug,:,:])])
            global_max = np.max([global_max, np.nanmax(pi[idxdrug,:,:])])
        from pathlib import Path

        Path(f"/data/users/quentin/package_paper/experiments/figures/{COHORT}/survival_per_drug_{gene_set}/").mkdir(parents=True, exist_ok=True)

        savefig = f"/data/users/quentin/package_paper/experiments/figures/{COHORT}/survival_per_drug_{gene_set}/{drug_name}.png"
        survival_probabilities(data_ref, copy.deepcopy(pi), model.cluster2clonelabel, df_info_cohort=model.dfinfo, idxdrug=idxdrug, drug_name=drug_name, savefig=savefig)
    

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

def survival_probabilities(data, piorig, cluster2clonelabel, df_info_cohort=None, idxdrug=0, label_colors=None, drug_name=None,  savefig=None):
    # Comparing true survival probabilities and the one estimated
    label2name = {'healthy':'Non-malignant', 'tumor':'Tumour', 'putative':'Putative'}

    pi = piorig.copy()
    for d in range(data['D']):
        pi[d,:,:][~(data['masks']['RNA'])] = float('nan')

    # Sort clusters based on their labels
    sorted_indices = np.argsort(cluster2clonelabel)

    # sorted_indices = np.arange(len(cluster2clonelabel))  #np.argsort(cluster2clonelabel)
    sorted_pi = pi[idxdrug, sorted_indices, :]

    if not(df_info_cohort is None):
        # Sort samples by patient_id and tissue_type
        sample_names = df_info_cohort.index.values
        sorted_meta_data = df_info_cohort.sort_values(by=['patient_id', 'tissue_type'])
        sorted_sample_ids = sorted_meta_data.index
        sample2idx = {sample: idx for idx, sample in enumerate(sorted_sample_ids.values)}
        sorted_indices_patient = np.array([sample2idx[sample] for sample in sample_names])
        sorted_pi = sorted_pi[:, sorted_indices_patient]

    # Get unique cluster labels and assign a color to each
    unique_labels = np.unique(cluster2clonelabel)
    if label_colors is None:
        label_colors = plt.colormaps['tab20'](np.linspace(0, 1, len(unique_labels)))

    # Create a color array for the y-axis labels, formatted for display
    row_colors = np.array([label_colors[np.where(unique_labels == label)[0][0]] for label in np.array(cluster2clonelabel)[sorted_indices]])

    # Reshape to a 2D array of RGB colors (n_clusters, 3) -> (n_clusters, 1, 3) for displaying as an image
    row_colors = row_colors.reshape(-1, 1, 4)  # Change 3 to 4 if the color has an alpha channel (RGBA)



    if not(df_info_cohort is None):
        # Create the main plot with two subplots: one for the color bar and one for the heatmap
        fig = plt.figure(layout="constrained")
        gs = GridSpec(2, 2, figure=fig, width_ratios=[0.05, 1], height_ratios=[1, 0.05])
        ax1 = fig.add_subplot(gs[0, 0])
        ax2 = fig.add_subplot(gs[0, 1])
    else:
        # Create the main plot with two subplots: one for the color bar and one for the heatmap
        fig, (ax1, ax2) = plt.subplots(ncols=2, gridspec_kw={"width_ratios": [0.005, 1]}, figsize=(12, 8))

    # Plot the color bar on the left using the label colors
    ax1.imshow(row_colors, aspect='auto', interpolation='nearest')
    ax1.axis('off')  # Remove the axis for the color bar

    # Plot the heatmap using seaborn with consistent vmin and vmax across all plots
    sns.heatmap(sorted_pi, cmap='viridis', cbar=True, ax=ax2, yticklabels=False, linewidths=0.5, linecolor='lightgray', vmin=global_min, vmax=global_max)

    # Set the y-axis labels to the sorted cluster indices
    ax2.set_yticks(np.arange(len(sorted_indices)) + 0.5)

    # Custom legend for cluster labels
    label2clustername = {}
    for label in label2name.keys():
        if COHORT=='aml':
            label2clustername[label] = 'clusters'
        else:
            label2clustername[label] = 'clones'
    label2clustername['healthy'] = 'cells'
    handles = [mpatches.Patch(color=label_colors[i], label=f'{label2name[label]} {label2clustername[label]}') for i, label in enumerate(unique_labels)]
    if COHORT=='aml':
        ax2.legend(handles=handles, bbox_to_anchor=(0.5, 1.1), loc='upper center', borderaxespad=0., ncols=1)
    else:
        ax2.legend(handles=handles, bbox_to_anchor=(0.5, 1.09), loc='upper center', borderaxespad=0., ncols=1)
    ax2.set_xticks([])

    # Optional: If you have patient labels
    # ax_heatmap.set_xticks(ticks=np.arange(sorted_pi.shape[1]))
    # ax_heatmap.set_xticklabels(patient_labels, rotation=90)

    # Show the plot
    if not(drug_name is None):
        if drug_name == 'empty':
            pass
        else:
            plt.suptitle(f"Heatmap for the drug: {drug_name}")
    else:
        plt.suptitle(f"Heatmap for Drug Index {idxdrug}")

    # Optional: Save the figure
    if savefig:
        plt.savefig(savefig, dpi=250, bbox_inches='tight')

    # Display the plot
    plt.show()



In [ ]:

import copy
#idxdrug = np.where(model.FD.selected_drugs == 'Alisertib')[0][0]
idxdrug = np.where(model.FD.selected_drugs=='Dabrafenib')[0][0]
global_min = 1
global_max = 1
for gene_set, pi in gene_set2pi.items():
    global_min = np.min([global_min, np.nanmin(pi[idxdrug,:,:])])
    global_max = np.max([global_max, np.nanmax(pi[idxdrug,:,:])])
for gene_set, pi in gene_set2pi.items():
    print(gene_set)
    if COHORT=='aml':
        survival_probabilities(data_ref, copy.deepcopy(pi), model.cluster2clonelabel, global_min, global_max, df_info_cohort=model.dfinfo, idxdrug=idxdrug, drug_name='empty', savefig='aml_{1}_{0}.png'.format(model.FD.selected_drugs[idxdrug], gene_set))
    else:
        survival_probabilities(data_ref, copy.deepcopy(pi), model.cluster2clonelabel, global_min, global_max, idxdrug=idxdrug, drug_name='empty', savefig='mel_{1}_{0}.png'.format(model.FD.selected_drugs[idxdrug], gene_set))


In [ ]:
model.FD.selected_drugs

In [ ]:
# Stratifying by samples
pi = np.nan_to_num(deepcopy(res['pi'].detach().numpy()))
ratio_pi = deepcopy(pi)
for idxdrug in range(pi.shape[0]):
    for sampleid in range(pi.shape[2]):
        learned_props = params_svi['proportions'][sampleid,:]
        healthy_survival =  np.sum((pi[idxdrug,:,sampleid]*learned_props)[model.cat2clusters['healthy']])
        healthy_survival /=  np.sum(learned_props[model.cat2clusters['healthy']])
        ratio_pi[idxdrug,:,sampleid] /= healthy_survival

for i in range(1,3):
    sccdr.resultanalysis.survival_probabilities_relative_by_patient(data_ref, ratio_pi, res['pi'].detach().numpy(), model.cluster2clonelabel, model.cluster2cat, model.FD.selected_drugs, sampleID=i)

In [ ]:
for COHORT in ['aml']:
    for gene_set_collection in gene_set_collections:
        for clonemode in cohort2clonemodes[COHORT]:

            mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)

            with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
                params_svi = pickle.load(handle)

            import h5py
            with h5py.File(os.path.join(rootpath, COHORT, gene_set_collection+'_'+clonemode, 'local_importance.h5'), "r") as f:
                # Print all root level object names (aka keys) 
                # these can be group or dataset names 
                print("Keys: %s" % f.keys())
                # get first object name/key; may or may NOT be a group
                print(list(f['local_importance_mean'].shape))
                print(f['local_importance_mean'].attrs.keys())
                drugs = f['local_importance_mean'].attrs['dim1_drugs']
                dims = f['local_importance_mean'].attrs['dim4_dimensions']


            filepath = os.path.join(rootpath,'{0}/{1}_{2}/beta.h5'.format(COHORT, gene_set_collection, clonemode))
            with h5py.File(filepath, 'w') as f:
                # Create a dataset
                dset = f.create_dataset('beta', data=params_svi['beta'])

                column_names = {
                'dim2_dimensions': dims,
                'dim1_drugs': drugs
                }

                for dim, labels in column_names.items():
                    dset.attrs[dim] = labels

In [ ]:

import sys
sys.path.append('/data/users/quentin/final_package/')
import scClone2DR as sccdr
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
import pickle
import torch
from tqdm import tqdm
import os
import plotly.io as pio
rootpath = '/data/users/quentin/final_package/experiments/paper_results'

# gene_set_collections = ['c6','hallmarks', 'c2_pid']
# cohort2clonemodes = {'aml':['phenograph'], 'melanoma':['scatrex']}
for COHORT in ['aml']:
    for gene_set_collection in gene_set_collections:
        for clonemode in cohort2clonemodes[COHORT]:

            mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)
            if COHORT=='melanoma':
                path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
                path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
                concentration_DMSO = '100'
                concentration_drug = '5'
                path_info_cohort = None
            elif COHORT=='aml':
                concentration_DMSO = '200'
                concentration_drug = '10'
                path_rna = '/data/users/04_share_reanalysis_results/aml_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
                path_fastdrug = '/data/users/04_share_reanalysis_results/01_aml/AML_PCY_cell_numbers_no_plate_effect_correction.csv'
                path_info_cohort = '/data/users/04_share_reanalysis_results/01_aml/2024-08-15_aml_overview_scRNA.tsv'

            model = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort)
            data_ref = model.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)


            with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
                params_svi = pickle.load(handle)

            with open(os.path.join(rootpath,'{0}/{1}_{2}/posterior_mean_params.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
                res = pickle.load(handle)

            pi = np.nan_to_num(deepcopy(res['pi'].detach().numpy()))
            ratio_pi = deepcopy(pi)
            for idxdrug in range(pi.shape[0]):
                for sampleid in range(pi.shape[2]):
                    learned_props = params_svi['proportions'][sampleid,:]
                    healthy_survival =  np.sum((pi[idxdrug,:,sampleid]*learned_props)[model.cat2clusters['healthy']])
                    healthy_survival /=  np.sum(learned_props[model.cat2clusters['healthy']])
                    ratio_pi[idxdrug,:,sampleid] /= healthy_survival

            import pandas as pd
            # N x Kmax
            props = params_svi['proportions']

            # ratio_pi: D x Kmax x N

            scores = np.sum(np.nan_to_num(ratio_pi)[:,model.cat2clusters['tumor'],:] * ((props[:,model.cat2clusters['tumor']].T)[None,:,:]), axis=1)
            scores /= np.tile((np.sum(props[:,model.cat2clusters['tumor']], axis=1)).reshape(1,-1), (ratio_pi.shape[0], 1))

            dic = {'sampleID': model.sample_names}
            for idxdrug, drug in enumerate(model.FD.selected_drugs):
                dic[drug] = scores[idxdrug,:]

            df = pd.DataFrame(dic)
            df.set_index('sampleID', inplace=True)

            df.to_csv(os.path.join(rootpath,'{0}/{1}_{2}/sample_drug_tumor_survival_probabilities.csv'.format(COHORT, gene_set_collection, clonemode)))

In [ ]:
for COHORT in ['aml']:
    for gene_set_collection in gene_set_collections:
        for clonemode in cohort2clonemodes[COHORT]:

            mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)
            if COHORT=='melanoma':
                path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
                path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
                concentration_DMSO = '100'
                concentration_drug = '5'
                path_info_cohort = None
            elif COHORT=='aml':
                concentration_DMSO = '200'
                concentration_drug = '10'
                path_rna = '/data/users/04_share_reanalysis_results/aml_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
                path_fastdrug = '/data/users/04_share_reanalysis_results/01_aml/AML_PCY_cell_numbers_no_plate_effect_correction.csv'
                path_info_cohort = '/data/users/04_share_reanalysis_results/01_aml/2024-08-15_aml_overview_scRNA.tsv'
            dfinfos = pd.read_csv(os.path.join(path_rna, "clone_infos.csv"), index_col='cloneID')
            dfinfos.to_csv(os.path.join(rootpath,'{0}/{1}_{2}/clone_infos.csv'.format(COHORT, gene_set_collection, clonemode)))

# Fold change

In [ ]:
model.get_fold_change(data_ref, params_svi, DIC_sample=res)

In [ ]:
model_bulk = sccdr.models.scClone2DR()

model_bulk = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort)
data_ref = model_bulk.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)

databulk = model_bulk.get_bulk_from_real_data(data_ref, model.cluster2cat, model.cat2clusters)
model_bulk.data = databulk
idxs_train = range(model.N)
idxs_test = []
data_train_bulk, data_test_bulk, sample_names_train_bulk, sample_names_test_bulk = model_bulk.get_real_data_split(idxs_train, idxs_test)

params_svi_bulk = model_bulk.train(data_train_bulk, penalty_l1=0.1, penalty_l2=0.1 , n_steps=2000)

In [ ]:
postmeans = model_bulk.get_posterior_mean_latentvar(params_svi_bulk, nsamples=100)
params_svi_bulk.update(postmeans)
dd = model_bulk.merge_data_param(data_train_bulk, {key:val for key, val in params_svi_bulk.items() if not('sample' in key)})
samples = model_bulk.sampling(dd)

model_bulk.get_fold_change(data_train_bulk, params_svi_bulk, DIC_sample=samples)

In [ ]:
model_bimodal = sccdr.models.scClone2DR()

model_bimodal = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort)
data_ref = model_bimodal.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)

databimodal = model_bimodal.get_bimodal_from_real_data(data_ref, model.cluster2cat, model.cat2clusters)
model_bimodal.data = databimodal
idxs_train = range(model.N)
idxs_test = []
data_train_bimodal, data_test_bimodal, sample_names_train_bimodal, sample_names_test_bimodal = model_bimodal.get_real_data_split(idxs_train, idxs_test)

params_svi_bimodal = model_bimodal.train(data_train_bimodal, penalty_l1=0.1, penalty_l2=0.1 , n_steps=1000)


postmeans = model_bimodal.get_posterior_mean_latentvar(params_svi_bimodal, nsamples=100)
params_svi_bimodal.update(postmeans)
dd = model_bimodal.merge_data_param(data_train_bimodal, {key:val for key, val in params_svi_bimodal.items() if not('sample' in key)})
samples = model_bimodal.sampling(dd)

model_bimodal.get_fold_change(data_train_bimodal, params_svi_bimodal, DIC_sample=samples)

In [ ]:
res = {}
res['us'] = {}
res['us'] = model.results
res['bulk'] = model_bulk.results
res['bimodal'] = model_bimodal.results

import seaborn as sns
colors_models = sns.color_palette('Set2')

models = ['us', 'base', 'bimodal','bulk','fm','fm_true_props','nn','nn_true_props']
colors = {m:colors_models[i] for i,m in enumerate((models))}

models = ['us', 'bimodal','bulk']
model2name = {'us':'scClone2DR', 'base':'Base', 'bimodal':'Bimodal','bulk':'Bulk','fm':'FM','nn':'NN', 
              'fm_true_props': 'FM true props','nn_true_props':'NN true props'}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import GridSpec

def corr_filtered(fc_true, fc_pred):
    fc_true = np.array(fc_true)
    fc_pred = np.array(fc_pred)
    # Create a boolean mask for valid values
    mask = (
        ~np.isnan(fc_true) & 
        ~np.isnan(fc_pred) & 
        (np.abs(fc_true) <= 100) & 
        (np.abs(fc_pred) <= 100)
    )

    # Apply mask and compute R²
    corr = np.round(np.corrcoef(fc_true[mask], fc_pred[mask])[0, 1]**2, 3)
    
    # Filtered data
    x_filtered = fc_true[mask]
    y_filtered = fc_pred[mask]

    # Get min values
    min_x = np.min(x_filtered)
    min_y = np.min(y_filtered)
    m = min([min_x, min_y])
    return m, corr

size_dots = 2
tit = ['This paper', 'Bimodal model', 'Bulk model']
# Sample data (replace with your actual data)
#colors = res['us']['colors']
colors = {m:colors_models[i] for i,m in enumerate((models))}

# Create subplots
fig = plt.figure(figsize=(18, 8))
gs = GridSpec(1, 3, width_ratios=[1, 1, 1])

custom_fontsize_labelsaxis, custom_font_size = 16,16

# Plot for model 1
ax00 = fig.add_subplot(gs[0, 0])
fc_true = res['us']['fold_change_data']
fc_pred = res['us']['fold_change_pred']
ax00.scatter(fc_true, fc_pred, color=colors['us'], s=size_dots)
ax00.set_title('{0}'.format(tit[0]), fontsize=custom_font_size)
ax00.set_xlabel('Fold change (observed)', fontsize=custom_fontsize_labelsaxis)
ax00.set_ylabel('Fold change (predicted)', fontsize=custom_fontsize_labelsaxis)
ax00.axis('square')
m, corr = corr_filtered(fc_true, fc_pred)
ax00.text(m+0.02,m+0.1, "Explained variance: {0}".format(corr), fontsize=14)

ax01 = fig.add_subplot(gs[0, 1])
fc_true = res['us']['fold_change_data']
fc_pred = res['bulk']['fold_change_pred']
ax01.scatter(fc_true, fc_pred, color=colors['bulk'], s=size_dots)
ax01.set_title('{0}'.format(tit[1]), fontsize=custom_font_size)
ax01.set_xlabel('Fold change (observed)', fontsize=custom_fontsize_labelsaxis)
ax01.set_ylabel('Fold change (predicted)', fontsize=custom_fontsize_labelsaxis)
ax01.axis('square')
m, corr = corr_filtered(fc_true, fc_pred)
ax01.text(m+0.1,m+0.1, "Explained variance: {0}".format(corr), fontsize=14)


ax10 = fig.add_subplot(gs[0, 2])
fc_true = res['us']['fold_change_data']
fc_pred = res['bimodal']['fold_change_pred']
ax10.scatter(fc_true, fc_pred, color=colors['bimodal'], s=size_dots)
ax10.set_title('{0}'.format(tit[2]), fontsize=custom_font_size)
ax10.set_xlabel('Fold change (observed)', fontsize=custom_fontsize_labelsaxis)
ax10.set_ylabel('Fold change (predicted)', fontsize=custom_fontsize_labelsaxis)
ax10.axis('square')
m, corr = corr_filtered(fc_true, fc_pred)
ax10.text(m+0.1,m+0.02, "Explained variance: {0}".format(corr), fontsize=14)
# ax10.set_ylim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
# ax10.set_xlim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))

plt.savefig('/data/users/quentin/final_package/experiments/paper_results/fold_change_real_data.png', dpi=250, bbox_inches='tight')


# Fold change test data

In [ ]:
idxs_train = [i for i in range(int(0.6 * model.N))]
idxs_test = [i for i in range(model.N) if not(i in idxs_train)]
data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, idxs_test)

params = model.train(data_train, penalty_l1=3.2, penalty_l2=3.2 , n_steps=2000)

params['proportions'] = params['proportions'][len(idxs_train):,:]
params['theta_fd'] = params['theta_fd'][len(idxs_train):]
postmeans = model.get_posterior_mean_latentvar(params, nsamples=100)
params.update(postmeans)
res = model.sampling(data_test, params)

In [ ]:
model.get_fold_change(data_test, params, DIC_sample=res)

In [ ]:
model_bulk = sccdr.models.scClone2DR()

model_bulk = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort)
data_ref = model_bulk.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)
databulk = model_bulk.get_bulk_from_real_data(data_ref, model.cluster2cat, model.cat2clusters)
model_bulk.data = databulk
data_train_bulk, data_test_bulk, sample_names_train_bulk, sample_names_test_bulk = model_bulk.get_real_data_split(idxs_train, idxs_test)
params_svi_bulk = model_bulk.train(data_train_bulk, penalty_l1=0.1, penalty_l2=0.1 , n_steps=2000)

In [ ]:
params_svi_bulk['proportions'] = params_svi_bulk['proportions'][len(idxs_train):,:]
params_svi_bulk['theta_fd'] = params_svi_bulk['theta_fd'][len(idxs_train):]
postmeans = model_bulk.get_posterior_mean_latentvar(params_svi_bulk, nsamples=100)
params_svi_bulk.update(postmeans)
dd = model_bulk.merge_data_param(data_test_bulk, {key:val for key, val in params_svi_bulk.items() if not('sample' in key)})
samples = model_bulk.sampling(dd)
model_bulk.get_fold_change(data_test_bulk, params_svi_bulk, DIC_sample=samples)

In [ ]:
model_bimodal = sccdr.models.scClone2DR()

model_bimodal = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort)
data_ref = model_bimodal.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)
databimodal = model_bimodal.get_bimodal_from_real_data(data_ref, model.cluster2cat, model.cat2clusters)
model_bimodal.data = databimodal
data_train_bimodal, data_test_bimodal, sample_names_train_bimodal, sample_names_test_bimodal = model_bimodal.get_real_data_split(idxs_train, idxs_test)

params_svi_bimodal = model_bimodal.train(data_train_bimodal, penalty_l1=0.1, penalty_l2=0.1 , n_steps=1000)

params_svi_bimodal['proportions'] = params_svi_bimodal['proportions'][len(idxs_train):,:]
params_svi_bimodal['theta_fd'] = params_svi_bimodal['theta_fd'][len(idxs_train):]
postmeans = model_bimodal.get_posterior_mean_latentvar(params_svi_bimodal, nsamples=100)
params_svi_bimodal.update(postmeans)
dd = model_bimodal.merge_data_param(data_test_bimodal, {key:val for key, val in params_svi_bimodal.items() if not('sample' in key)})
samples = model_bimodal.sampling(dd)

model_bimodal.get_fold_change(data_test_bimodal, params_svi_bimodal, DIC_sample=samples)

In [ ]:
res = {}
res['us'] = {}
res['us'] = model.results
res['bulk'] = model_bulk.results
res['bimodal'] = model_bimodal.results

import seaborn as sns
colors_models = sns.color_palette('Set2')

models = ['us', 'base', 'bimodal','bulk','fm','fm_true_props','nn','nn_true_props']
colors = {m:colors_models[i] for i,m in enumerate((models))}

models = ['us', 'bimodal','bulk']
model2name = {'us':'scClone2DR', 'base':'Base', 'bimodal':'Bimodal','bulk':'Bulk','fm':'FM','nn':'NN', 
              'fm_true_props': 'FM true props','nn_true_props':'NN true props'}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import GridSpec

def corr_filtered(fc_true, fc_pred):
    fc_true = np.array(fc_true)
    fc_pred = np.array(fc_pred)
    # Create a boolean mask for valid values
    mask = (
        ~np.isnan(fc_true) & 
        ~np.isnan(fc_pred) & 
        (np.abs(fc_true) <= 100) & 
        (np.abs(fc_pred) <= 100)
    )

    # Apply mask and compute R²
    corr = np.round(np.corrcoef(fc_true[mask], fc_pred[mask])[0, 1]**2, 3)
    
    # Filtered data
    x_filtered = fc_true[mask]
    y_filtered = fc_pred[mask]

    # Get min values
    min_x = np.min(x_filtered)
    min_y = np.min(y_filtered)
    m = min([min_x, min_y])
    return m, corr

size_dots = 2
tit = ['This paper', 'Bimodal model', 'Bulk model']
# Sample data (replace with your actual data)
#colors = res['us']['colors']
colors = {m:colors_models[i] for i,m in enumerate((models))}

# Create subplots
fig = plt.figure(figsize=(18, 8))
gs = GridSpec(1, 3, width_ratios=[1, 1, 1])

custom_fontsize_labelsaxis, custom_font_size = 16,16

# Plot for model 1
ax00 = fig.add_subplot(gs[0, 0])
fc_true = res['us']['fold_change_data']
fc_pred = res['us']['fold_change_pred']
ax00.scatter(fc_true, fc_pred, color=colors['us'], s=size_dots)
ax00.set_title('{0}'.format(tit[0]), fontsize=custom_font_size)
ax00.set_xlabel('Fold change (observed)', fontsize=custom_fontsize_labelsaxis)
ax00.set_ylabel('Fold change (predicted)', fontsize=custom_fontsize_labelsaxis)
ax00.axis('square')
m, corr = corr_filtered(fc_true, fc_pred)
#ax00.text(m+0.02,m+0.1, "Explained variance: {0}".format(corr), fontsize=14)

ax01 = fig.add_subplot(gs[0, 1])
fc_true = res['us']['fold_change_data']
fc_pred = res['bulk']['fold_change_pred']
ax01.scatter(fc_true, fc_pred, color=colors['bulk'], s=size_dots)
ax01.set_title('{0}'.format(tit[1]), fontsize=custom_font_size)
ax01.set_xlabel('Fold change (observed)', fontsize=custom_fontsize_labelsaxis)
ax01.set_ylabel('Fold change (predicted)', fontsize=custom_fontsize_labelsaxis)
ax01.axis('square')
m, corr = corr_filtered(fc_true, fc_pred)
#ax01.text(m+0.1,m+0.1, "Explained variance: {0}".format(corr), fontsize=14)


ax10 = fig.add_subplot(gs[0, 2])
fc_true = res['us']['fold_change_data']
fc_pred = res['bimodal']['fold_change_pred']
ax10.scatter(fc_true, fc_pred, color=colors['bimodal'], s=size_dots)
ax10.set_title('{0}'.format(tit[2]), fontsize=custom_font_size)
ax10.set_xlabel('Fold change (observed)', fontsize=custom_fontsize_labelsaxis)
ax10.set_ylabel('Fold change (predicted)', fontsize=custom_fontsize_labelsaxis)
ax10.axis('square')
m, corr = corr_filtered(fc_true, fc_pred)
#ax10.text(m+0.1,m+0.02, "Explained variance: {0}".format(corr), fontsize=14)
# ax10.set_ylim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
# ax10.set_xlim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))

plt.savefig('/data/users/quentin/final_package/experiments/paper_results/fold_change_real_data.png', dpi=250, bbox_inches='tight')


In [ ]:
fold_changes_pred = []
fold_changes_notpred = []

learned_props = np.zeros((len(model.sample_names), model.Kmax))

res = get_res_from_sample(model, params_svi, data_ref)
ls_vars = ['n0_c', 'n0_r', 'n_r', 'n_c', 'n_rna', 'pi', 'nu_healthy_drug', 'nu_healthy_control']
for var in ls_vars:
    res[var] = torch.zeros(res[var].shape)


for id_patient in tqdm(range(len(model.sample_names))):
    patient = model.sample_names[id_patient]

    idxs_train = [i for i in range(int(data_ref['N'])) if i!=id_patient]
    idxs_test = [id_patient]
    data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, idxs_test)
    params_svi_validation = {}
    for key, val in params_svi.items():
        if torch.is_tensor(val):
            params_svi_validation[key] = val.clone().detach()
        else:
            params_svi_validation[key] = val
    params_svi_validation['proportions'] = params_svi_validation['proportions'][[len(idxs_train)],:]
    params_svi_validation['theta_fd'] = params_svi_validation['theta_fd'][[len(idxs_train)]]
    data_validation = deepcopy(data_test)

    learned_props[id_patient,:] = params_svi['proportions'][len(idxs_train),:]

    if True:
        nb_ites = 100
        for ite in range(nb_ites):
            restmp = get_res_from_sample(model, params_svi_validation, data_validation)
            res['n0_c'][:,id_patient] += restmp['n0_c'][:,0]/nb_ites
            res['n_c'][:,id_patient] += restmp['n_c'][:,0]/nb_ites
            res['n0_r'][:,:,id_patient] += restmp['n0_r'][:,:,0]/nb_ites
            res['n_rna'][:,id_patient] += restmp['n_rna'][:,0]/nb_ites
            res['pi'][:,:,id_patient] += restmp['pi'][:,:,0]/nb_ites
            res['nu_healthy_drug'][:,:,id_patient] += restmp['nu_healthy_drug'][:,:,0]/nb_ites
            res['nu_healthy_control'][:,id_patient] += restmp['nu_healthy_control'][:,0]/nb_ites
    else:
        nb_ites = 100
        from torch.distributions import MultivariateNormal
        loc = params_svi_validation['AutoMultivariateNormal.loc']
        
        # Define the scale_tril (lower triangular matrix)
        scale_tril = params_svi_validation['AutoMultivariateNormal.scale_tril']
        
        # Define the multivariate Gaussian distribution
        mvn = MultivariateNormal(loc=torch.tensor(loc), scale_tril=torch.tensor(scale_tril))
        gamma = mvn.sample()
        dim = int(len(gamma)/model.n_clonelabels)
        for i, clonelabel in enumerate(model.clonelabels):
            params_svi_validation['gamma_{0}'.format(clonelabel)] = torch.zeros(gamma[i*dim:(i+1)*dim].shape)
        for ite in range(nb_ites):
            gamma = mvn.sample()
            dim = int(len(gamma)/model.n_clonelabels)
            for i, clonelabel in enumerate(model.clonelabels):
                params_svi_validation['gamma_{0}'.format(clonelabel)] += gamma[i*dim:(i+1)*dim]/nb_ites
                
        restmp = get_res_from_sample(model, params_svi_validation, data_validation)
        res['n0_c'][:,id_patient] = restmp['n0_c'][:,0]
        res['n_c'][:,id_patient] = restmp['n_c'][:,0]
        res['n0_r'][:,:,id_patient] = restmp['n0_r'][:,:,0]
        res['n_rna'][:,id_patient] = torch.tensor(restmp['n_rna'][:,0])
        res['pi'][:,:,id_patient] = restmp['pi'][:,:,0]
        res['nu_healthy_drug'][:,:,id_patient] = restmp['nu_healthy_drug'][:,:,0]
        res['nu_healthy_control'][:,id_patient] = restmp['nu_healthy_control'][:,0]

    params_fold_change = {'pi': res['pi'][:,:,[id_patient]],
                          'nu_healthy_drug': res['nu_healthy_drug'][:,:,[id_patient]],
                          'nu_healthy_control': res['nu_healthy_control'][:,[id_patient]],
                          'proportions': params_svi_validation['proportions']
                        }
    fold_changes, pi, colors = model.get_fold_change(data_validation, params_fold_change, output_results=True)
    fold_changes_pred += list(fold_changes['pred'])
    fold_changes_notpred += list(fold_changes['not pred'])

    # fold_changes, pi, colors = model.get_fold_change(data_validation, params_svi_validation, output_results=True)
    # fold_changes_pred_test.append(fold_changes['pred'])
    # fold_changes_notpred_test.append(fold_changes['not pred'])

In [ ]:
clonemodetemps = {'melanoma': ['scatrex','phenograph'], 'aml': ['phenograph']}
for COHORT in ['melanoma','aml']:
    for gene_set_collection in ['geneOncoKB']:
        for clonemode in clonemodetemps[COHORT]:

            mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)

            with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
                params_svi = pickle.load(handle)

            import h5py
            with h5py.File(os.path.join(rootpath, COHORT, gene_set_collection+'_'+clonemode, 'local_importance.h5'), "r") as f:
                # Print all root level object names (aka keys) 
                # these can be group or dataset names 
                print("Keys: %s" % f.keys())
                # get first object name/key; may or may NOT be a group
                print(list(f['local_importance_mean'].shape))
                print(f['local_importance_mean'].attrs.keys())
                drugs = f['local_importance_mean'].attrs['dim1_drugs']
                dims = f['local_importance_mean'].attrs['dim4_dimensions']


            filepath = os.path.join(rootpath,'{0}/{1}_{2}/beta.h5'.format(COHORT, gene_set_collection, clonemode))
            with h5py.File(filepath, 'w') as f:
                # Create a dataset
                dset = f.create_dataset('beta', data=params_svi['beta'])

                column_names = {
                'dim2_dimensions': dims,
                'dim1_drugs': drugs
                }

                for dim, labels in column_names.items():
                    dset.attrs[dim] = labels

In [ ]:

import sys
sys.path.append('/data/users/quentin/final_package/')
import scClone2DR as sccdr
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
import pickle
import torch
from tqdm import tqdm
import os
import plotly.io as pio
rootpath = '/data/users/quentin/final_package/experiments/paper_results'

clonemodetemps = {'melanoma': ['scatrex','phenograph'], 'aml': ['phenograph']}
for COHORT in ['melanoma','aml']:
    for gene_set_collection in ['geneOncoKB']:
        for clonemode in clonemodetemps[COHORT]:
            
            if 'sparser' in clonemode:
                mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode[:-8])
            else:
                mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)
            if COHORT=='melanoma':
                path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
                path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
                concentration_DMSO = '100'
                concentration_drug = '5'
                path_info_cohort = None
            elif COHORT=='aml':
                concentration_DMSO = '200'
                concentration_drug = '10'
                path_rna = '/data/users/04_share_reanalysis_results/aml_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
                path_fastdrug = '/data/users/04_share_reanalysis_results/01_aml/AML_PCY_cell_numbers_no_plate_effect_correction.csv'
                path_info_cohort = '/data/users/04_share_reanalysis_results/01_aml/2024-08-15_aml_overview_scRNA.tsv'

            model = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort)
            data_ref = model.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)


            with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
                params_svi = pickle.load(handle)

            with open(os.path.join(rootpath,'{0}/{1}_{2}/posterior_mean_params.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
                res = pickle.load(handle)

            pi = np.nan_to_num(deepcopy(res['pi'].detach().numpy()))
            ratio_pi = deepcopy(pi)
            for idxdrug in range(pi.shape[0]):
                for sampleid in range(pi.shape[2]):
                    learned_props = params_svi['proportions'][sampleid,:]
                    healthy_survival =  np.sum((pi[idxdrug,:,sampleid]*learned_props)[model.cat2clusters['healthy']])
                    healthy_survival /=  np.sum(learned_props[model.cat2clusters['healthy']])
                    ratio_pi[idxdrug,:,sampleid] /= healthy_survival

            import pandas as pd
            # N x Kmax
            props = params_svi['proportions']

            # ratio_pi: D x Kmax x N

            scores = np.sum(np.nan_to_num(ratio_pi)[:,model.cat2clusters['tumor'],:] * ((props[:,model.cat2clusters['tumor']].T)[None,:,:]), axis=1)
            scores /= np.tile((np.sum(props[:,model.cat2clusters['tumor']], axis=1)).reshape(1,-1), (ratio_pi.shape[0], 1))

            dic = {'sampleID': model.sample_names}
            for idxdrug, drug in enumerate(model.FD.selected_drugs):
                dic[drug] = scores[idxdrug,:]

            df = pd.DataFrame(dic)
            df.set_index('sampleID', inplace=True)

            df.to_csv(os.path.join(rootpath,'{0}/{1}_{2}/sample_drug_tumor_survival_probabilities.csv'.format(COHORT, gene_set_collection, clonemode)))

In [ ]:
clonemodetemps = {'melanoma': ['scatrex','phenograph'], 'aml': ['phenograph']}
for COHORT in ['melanoma','aml']:
    for gene_set_collection in ['geneOncoKB']:
        for clonemode in clonemodetemps[COHORT]:
            if 'sparser' in clonemode:
                mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode[:-8])
            else:
                mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)
            if COHORT=='melanoma':
                path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
                path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
                concentration_DMSO = '100'
                concentration_drug = '5'
                path_info_cohort = None
            elif COHORT=='aml':
                concentration_DMSO = '200'
                concentration_drug = '10'
                path_rna = '/data/users/04_share_reanalysis_results/aml_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
                path_fastdrug = '/data/users/04_share_reanalysis_results/01_aml/AML_PCY_cell_numbers_no_plate_effect_correction.csv'
                path_info_cohort = '/data/users/04_share_reanalysis_results/01_aml/2024-08-15_aml_overview_scRNA.tsv'
            dfinfos = pd.read_csv(os.path.join(path_rna, "clone_infos.csv"), index_col='cloneID')
            dfinfos.to_csv(os.path.join(rootpath,'{0}/{1}_{2}/clone_infos.csv'.format(COHORT, gene_set_collection, clonemode)))